# VirtualFit — IDM-VTON Try-On Server (Colab)

Hosts the **real IDM-VTON** virtual try-on model (the strong SDXL-inpainting model from
*"Improving Diffusion Models for Authentic Virtual Try-on in the Wild"*) on a Colab GPU and
exposes it as a FastAPI endpoint through a **cloudflared** quick tunnel.

It speaks the **exact same `/health` + `/tryon` contract** as the older CatVTON notebook and
**publishes to the same discovery store**, so it is a **drop-in replacement**: just run this
notebook instead of `catvton_server.ipynb` and the VirtualFit backend auto-discovers it — no
backend or `.env` changes needed.

Why this is more accurate than the old paths:
- Uses the **full IDM-VTON pipeline** (garment UNet encoder + IP-Adapter image encoder), which
  is higher fidelity than CatVTON and far more reliable than the public, often-queued HF Space.
- Builds a **proper agnostic mask** from **OpenPose keypoints + human-parsing (SCHP/ATR)** and a
  **DensePose** body map — instead of a generic auto-mask — so the garment lands in the right
  place and the face/identity/background are preserved.

## ⚠️ GPU requirement (read this)
IDM-VTON loads ~15 GB of fp16 weights and needs headroom for inference.
- **Best:** Colab **Pro** with an **L4 (24 GB)** or **A100** runtime → fast, no OOM.
- **Free T4 (16 GB):** this notebook auto-enables **model CPU-offload** when it detects <20 GB
  VRAM so it still fits, but each render is slower (~60–120 s). If you hit CUDA OOM on a T4,
  restart the runtime and re-run.

## How to run
1. **Runtime → Change runtime type → L4 / A100** (or T4), then **Run all** top-to-bottom.
2. First run downloads the IDM-VTON weights (~25 GB) — give it ~10–15 min.
3. The last cell prints the public URL **and publishes it**. VirtualFit picks it up automatically
   on the next try-on. Re-run the last cell whenever the Colab session restarts.

Endpoints: `GET /health`, `POST /tryon`.


In [ ]:
# 1. Confirm a GPU is attached and report VRAM (L4/A100 ideal; T4 works with offload).
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> GPU (L4/A100/T4)."
name = torch.cuda.get_device_name(0)
vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
print(f"CUDA: True | Device: {name} | VRAM: {vram} GB")
print("NOTE: <20 GB will auto-enable CPU offload (slower).") if vram < 20 else print("Plenty of VRAM.")


In [ ]:
# 2. Clone the official IDM-VTON repo (provides the custom pipeline, UNet hacks,
#    densepose/apply_net, human-parsing and openpose preprocessing, and configs).
%cd /content
!rm -rf IDM-VTON
!git clone https://github.com/yisol/IDM-VTON.git
%cd /content/IDM-VTON


In [ ]:
# 3. Install IDM-VTON deps for Colab (Python 3.12). torch 2.2.2 matches the
# late-2023 accelerate/transformers/peft stack (torch 2.4+ removed the dynamo
# GuardSource.LOCAL_NN_MODULE enum that accelerate 0.25 / peft reference).
# numpy MUST be <2 for detectron2. diffusers MUST be 0.25.0 for IDM-VTON's UNet.
# onnxruntime is CPU build (the -gpu wheel needs CUDA 13 -> libcudart.so.13 missing).
# AFTER this cell: Runtime > Restart session, then run cells 4, 5, 6.
import os
os.environ["FORCE_CUDA"] = "1"   # build detectron2 CUDA ops for the attached GPU
!pip -q install torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121 2>&1 | tail -3
!pip -q install "numpy==1.26.4" 2>&1 | tail -2
!pip -q install "git+https://github.com/facebookresearch/detectron2.git" 2>&1 | tail -8
!pip -q install "diffusers==0.25.0" "transformers==4.36.2" "accelerate==0.25.0" "peft==0.7.1" "huggingface_hub==0.20.3" onnxruntime "scipy==1.11.4" "einops==0.7.0" "torchmetrics==1.2.1" opencv-python "fvcore==0.1.5.post20221221" cloudpickle "omegaconf==2.3.0" pycocotools av fastapi "uvicorn[standard]" python-multipart 2>&1 | tail -6
print("INSTALL DONE - then Runtime > Restart session, run cells 4,5,6")

In [ ]:
# 4. Download IDM-VTON weights (~25 GB, first run only) and build the pipeline + preprocessors.
#    The HF repo yisol/IDM-VTON has densepose/humanparsing/openpose at top level; the repo code
#    expects them under ./ckpt/, so we symlink them. cwd MUST stay at the repo root.
import os, sys, torch
os.chdir("/content/IDM-VTON")
sys.path.insert(0, "/content/IDM-VTON")
# IDM-VTON ships apply_net.py, the densepose/ package, and utils_mask.py inside
# gradio_demo/. The official app.py works only because running it as a script puts
# gradio_demo/ on sys.path[0]; in a notebook we must add it explicitly (front).
sys.path.insert(0, "/content/IDM-VTON/gradio_demo")
from huggingface_hub import snapshot_download

BASE_PATH = snapshot_download(repo_id="yisol/IDM-VTON")
print("weights at:", BASE_PATH)

# Make ./ckpt/{densepose,humanparsing,openpose} point at the downloaded folders.
os.makedirs("ckpt", exist_ok=True)
for sub in ("densepose", "humanparsing", "openpose"):
    link = os.path.join("ckpt", sub)
    target = os.path.join(BASE_PATH, sub)
    if os.path.islink(link) or os.path.exists(link):
        try: os.remove(link)
        except IsADirectoryError: import shutil; shutil.rmtree(link)
    os.symlink(target, link)
    print("linked", link, "->", target)

from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
from src.unet_hacked_garmnet import UNet2DConditionModel as UNet2DConditionModel_ref
from src.unet_hacked_tryon import UNet2DConditionModel
from transformers import (CLIPImageProcessor, CLIPVisionModelWithProjection,
                          CLIPTextModel, CLIPTextModelWithProjection, AutoTokenizer)
from diffusers import DDPMScheduler, AutoencoderKL

dtype = torch.float16
device = "cuda"

unet = UNet2DConditionModel.from_pretrained(BASE_PATH, subfolder="unet", torch_dtype=dtype)
unet.requires_grad_(False)
UNet_Encoder = UNet2DConditionModel_ref.from_pretrained(BASE_PATH, subfolder="unet_encoder", torch_dtype=dtype)
UNet_Encoder.requires_grad_(False)
image_encoder = CLIPVisionModelWithProjection.from_pretrained(BASE_PATH, subfolder="image_encoder", torch_dtype=dtype)
vae = AutoencoderKL.from_pretrained(BASE_PATH, subfolder="vae", torch_dtype=dtype)
text_encoder_one = CLIPTextModel.from_pretrained(BASE_PATH, subfolder="text_encoder", torch_dtype=dtype)
text_encoder_two = CLIPTextModelWithProjection.from_pretrained(BASE_PATH, subfolder="text_encoder_2", torch_dtype=dtype)
tokenizer_one = AutoTokenizer.from_pretrained(BASE_PATH, subfolder="tokenizer", use_fast=False)
tokenizer_two = AutoTokenizer.from_pretrained(BASE_PATH, subfolder="tokenizer_2", use_fast=False)
noise_scheduler = DDPMScheduler.from_pretrained(BASE_PATH, subfolder="scheduler")

pipe = TryonPipeline.from_pretrained(
    BASE_PATH, unet=unet, vae=vae, feature_extractor=CLIPImageProcessor(),
    text_encoder=text_encoder_one, text_encoder_2=text_encoder_two,
    tokenizer=tokenizer_one, tokenizer_2=tokenizer_two,
    scheduler=noise_scheduler, image_encoder=image_encoder, torch_dtype=dtype,
)
pipe.unet_encoder = UNet_Encoder

# Preprocessors (human parsing + openpose). These load from ./ckpt/ (the symlinks above).
from preprocess.humanparsing.run_parsing import Parsing
from preprocess.openpose.run_openpose import OpenPose
parsing_model = Parsing(0)
openpose_model = OpenPose(0)

# Memory strategy: full GPU on big cards, CPU offload on small ones (T4) to avoid OOM.
TOTAL_VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9
LOW_VRAM = TOTAL_VRAM < 20
if LOW_VRAM:
    print(f"VRAM {TOTAL_VRAM:.1f} GB < 20 — enabling model CPU offload (slower, fits T4).")
    pipe.enable_model_cpu_offload()
else:
    print(f"VRAM {TOTAL_VRAM:.1f} GB — loading everything on GPU.")
    pipe.to(device)
    pipe.unet_encoder.to(device)
openpose_model.preprocessor.body_estimation.model.to(device)
print("PIPELINE + PREPROCESSORS READY")

In [ ]:
# 5. FastAPI server — same /health + /tryon contract as the CatVTON notebook, plus two
#    optional IDM-VTON fields (cloth_desc, auto_crop). Lock-serialized for a single GPU.
import threading, base64, io, time, numpy as np
from typing import List, Optional
from fastapi import FastAPI
from pydantic import BaseModel
from PIL import Image
import torch
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image

import apply_net
from utils_mask import get_mask_location
from detectron2.data.detection_utils import convert_PIL_to_numpy, _apply_exif_orientation

W, H = 768, 1024
tensor_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.5], [0.5])])
infer_lock = threading.Lock()

# Map the contract's cloth_type vocabulary to IDM-VTON mask categories.
CATEGORY = {"upper": "upper_body", "upper_body": "upper_body",
            "lower": "lower_body", "lower_body": "lower_body",
            "overall": "dresses", "dresses": "dresses", "full": "dresses"}

def b64_to_img(s):
    if "," in s and s.strip().startswith("data:"): s = s.split(",", 1)[1]
    return Image.open(io.BytesIO(base64.b64decode(s))).convert("RGB")

def img_to_b64(img):
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def pil_to_binary_mask(pil_image, threshold=0):
    arr = np.array(pil_image.convert("L"))
    mask = np.zeros(arr.shape, dtype=np.uint8)
    mask[arr > threshold] = 1
    return Image.fromarray((mask * 255).astype(np.uint8))

# Build the densepose argument parser once; reuse per request.
_dp_args = apply_net.create_argument_parser().parse_args((
    'show', './configs/densepose_rcnn_R_50_FPN_s1x.yaml',
    './ckpt/densepose/model_final_162be9.pkl', 'dp_segm', '-v',
    '--opts', 'MODEL.DEVICE', 'cuda'))

def run_idm_tryon(human_img_orig, garm_img, cloth_type, steps, guidance, seed,
                  garment_des, auto_crop):
    category = CATEGORY.get(cloth_type, "upper_body")
    garm_img = garm_img.resize((W, H))

    if auto_crop:
        width, height = human_img_orig.size
        target_width = int(min(width, height * (3 / 4)))
        target_height = int(min(height, width * (4 / 3)))
        left = (width - target_width) / 2; top = (height - target_height) / 2
        right = (width + target_width) / 2; bottom = (height + target_height) / 2
        cropped = human_img_orig.crop((left, top, right, bottom))
        crop_size = cropped.size
        human_img = cropped.resize((W, H))
    else:
        human_img = human_img_orig.resize((W, H))

    # Agnostic mask from openpose keypoints + human parsing.
    keypoints = openpose_model(human_img.resize((384, 512)))
    model_parse, _ = parsing_model(human_img.resize((384, 512)))
    mask, _ = get_mask_location('hd', category, model_parse, keypoints)
    mask = mask.resize((W, H))

    # DensePose body map.
    human_img_arg = _apply_exif_orientation(human_img.resize((384, 512)))
    human_img_arg = convert_PIL_to_numpy(human_img_arg, format="BGR")
    pose_img = _dp_args.func(_dp_args, human_img_arg)
    pose_img = pose_img[:, :, ::-1]
    pose_img = Image.fromarray(pose_img).resize((W, H))

    with torch.no_grad(), torch.cuda.amp.autocast():
        prompt = "model is wearing " + garment_des
        neg = "monochrome, lowres, bad anatomy, worst quality, low quality"
        with torch.inference_mode():
            (prompt_embeds, negative_prompt_embeds, pooled_prompt_embeds,
             negative_pooled_prompt_embeds) = pipe.encode_prompt(
                prompt, num_images_per_prompt=1,
                do_classifier_free_guidance=True, negative_prompt=neg)

        prompt_c = ["a photo of " + garment_des]
        neg_c = [neg]
        with torch.inference_mode():
            prompt_embeds_c, _, _, _ = pipe.encode_prompt(
                prompt_c, num_images_per_prompt=1,
                do_classifier_free_guidance=False, negative_prompt=neg_c)

        pose_t = tensor_transform(pose_img).unsqueeze(0).to(device, torch.float16)
        garm_t = tensor_transform(garm_img).unsqueeze(0).to(device, torch.float16)
        generator = torch.Generator(device).manual_seed(seed) if seed is not None else None

        images = pipe(
            prompt_embeds=prompt_embeds.to(device, torch.float16),
            negative_prompt_embeds=negative_prompt_embeds.to(device, torch.float16),
            pooled_prompt_embeds=pooled_prompt_embeds.to(device, torch.float16),
            negative_pooled_prompt_embeds=negative_pooled_prompt_embeds.to(device, torch.float16),
            num_inference_steps=steps, generator=generator, strength=1.0,
            pose_img=pose_t.to(device, torch.float16),
            text_embeds_cloth=prompt_embeds_c.to(device, torch.float16),
            cloth=garm_t.to(device, torch.float16),
            mask_image=mask, image=human_img, height=H, width=W,
            ip_adapter_image=garm_img.resize((W, H)),
            guidance_scale=guidance,
        )[0]

    out = images[0]
    if auto_crop:
        out = out.resize(crop_size)
        human_img_orig.paste(out, (int(left), int(top)))
        return human_img_orig
    return out

class TryOnReq(BaseModel):
    person_image: str
    cloth_image: str
    cloth_type: str = "upper"
    num_inference_steps: int = 30
    guidance_scale: float = 2.0
    seed: int = 42
    cloth_desc: Optional[str] = "a garment"      # IDM-VTON garment description (improves result)
    auto_crop: bool = True                        # center-crop to 3:4 for better alignment

app = FastAPI(title="VirtualFit IDM-VTON")

@app.get("/health")
def health():
    return {"status": "ok", "model": "IDM-VTON",
            "gpu": torch.cuda.get_device_name(0),
            "vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
            "low_vram_offload": bool(LOW_VRAM)}

@app.post("/tryon")
def tryon(req: TryOnReq):
    t0 = time.time()
    person = b64_to_img(req.person_image)
    cloth = b64_to_img(req.cloth_image)
    with infer_lock:
        result = run_idm_tryon(
            person, cloth, req.cloth_type, int(req.num_inference_steps),
            float(req.guidance_scale), int(req.seed),
            (req.cloth_desc or "a garment"), bool(req.auto_crop))
    return {"image": "data:image/png;base64," + img_to_b64(result),
            "seconds": round(time.time() - t0, 1)}

def _run():
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=_run, daemon=True).start()
time.sleep(5)
import requests
print(requests.get("http://localhost:8000/health").json())
print("SERVER UP on :8000")


In [ ]:
# 6. Tunnel + auto-publish URL to the SAME discovery store the backend already reads.
#    VirtualFit auto-discovers it — copy nothing. Re-run this cell after a session restart.
import subprocess, re, time, os, requests
# Must match backend .env TRYON_DISCOVERY_URL (same store the CatVTON notebook used).
DISCOVERY_URL = "https://jsonblob.com/api/jsonBlob/019ef86e-3c96-7d7f-8d36-e14639cf7546"

if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True); time.sleep(2)

logf = "/content/cf.log"
subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                 stdout=open(logf, "w"), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", open(logf).read())
    if m: PUBLIC_URL = m.group(0); break

for i in range(12):  # wait for tunnel DNS to propagate
    try:
        print("health:", requests.get(PUBLIC_URL + "/health", timeout=30).json()); break
    except Exception: time.sleep(5)

if PUBLIC_URL:
    try:
        requests.put(DISCOVERY_URL, headers={"Content-Type": "application/json"},
                     json={"url": PUBLIC_URL, "model": "IDM-VTON",
                           "updated": time.strftime("%Y-%m-%d %H:%M:%S")}, timeout=20)
        print("\n==> Published. VirtualFit will use this automatically on the next try-on.")
    except Exception as e:
        print("publish failed (set URL manually via /api/tryon/server):", e)
print("==> PUBLIC_URL:", PUBLIC_URL)
